# Deploy Product Agent to Amazon Bedrock AgentCore Runtime

This tutorial series builds a **3-agent e-commerce shopping assistant**. In this notebook, you deploy the Product Agent—an A2A server that searches product catalogs via external HTTP API and is invoked by a Orchestrator agent.

**Notebook 1 of 3** in the multi-agent orchestration series.

### What You'll Build

A **Product Agent** built with Strands Agents and deployed to Amazon Bedrock AgentCore that:
- Searches product catalogs via the DummyJSON API
- Responds to A2A protocol requests from other agents on port 9000
- Advertises its capabilities via an agent card for runtime discovery

### How You'll Build It

Using Strands Agents SDK and Amazon Bedrock AgentCore:
- Define HTTP API tools with the Strands Agents `@tool` decorator
- Wrap the agent with `A2AServer` for agent-to-agent communication
- Deploy to AgentCore runtime with IAM execution role
- Store the runtime URL in SSM Parameter Store for multi-agent discovery

## Architecture Overview

```
User Request
     │
     ▼
┌──────────────────┐
│   Orchestrator   │
│ (HTTP protocol)  │
└────────┬─────────┘
      │ A2A Protocol (SigV4 Auth)
      ├──────────────────┬
      ▼                  ▼
┌─────────────┐    ┌─────────────┐
│ Order Agent │    │Product Agent│  ◄── You are here (Notebook 1)
│ (A2A Server)│    │(A2A Server) │
└─────┬───────┘    └─────┬───────┘
      │                  │
      ▼                  ▼
┌─────────────┐    ┌─────────────┐
│  DynamoDB   │    │Product      │
│  (Orders)   │    │Catalog API  │
└─────────────┘    └─────────────┘
```

## Prerequisites

- [AWS CLI](https://aws.amazon.com/cli/) installed and configured
- Python 3.10 or higher
- Docker or Podman installed (only for local container builds; not required if using CodeBuild)
- Claude Sonnet 4 model access in Amazon Bedrock
- IAM permissions to:
  - Create IAM roles and policies
  - Create Amazon ECR repositories
  - Deploy Amazon Bedrock AgentCore runtimes
  - Write to AWS Systems Manager Parameter Store
- Local notebook dependencies: `pip install -r requirements.txt`

In [ ]:
# Install notebook dependencies
!pip install -r requirements.txt

In [ ]:
# Import standard libraries
import os
from pathlib import Path
from urllib.parse import quote
from uuid import uuid4

# Import AWS SDK
import boto3

# Import utilities
from utils import (
    PRODUCT_AGENT_NAME,
    PRODUCT_ROLE_NAME,
    SSM_PRODUCT_AGENT_URL,
    create_agentcore_role,
)
from utils.ssm_helpers import store_ssm_parameter

# Get AWS account information
sts = boto3.client("sts")
account_id = sts.get_caller_identity()["Account"]
region = "us-west-2"  # Multi-agent tutorial uses us-west-2

# Set paths
NOTEBOOK_DIR = Path.cwd()

# Ensure product_agent directory exists for %%writefile
(NOTEBOOK_DIR / "product_agent").mkdir(exist_ok=True)

print(f"AWS Account: {account_id}")
print(f"Region: {region}")

## Step 1: Create Product Agent with A2A Protocol

The Product Agent is a **specialist agent** that other agents call to perform product catalog searches. Unlike user-facing agents that receive HTTP requests directly, specialist agents use the **A2A protocol**—a standardized format for secure, structured communication between agents.

**Why A2A protocol for specialist agents?**

| Configuration | Purpose |
|---------------|---------|
| A2A protocol | Standardized JSON-RPC style messaging between agents (vs raw HTTP for users) |
| Port 9000 | Dedicated port for A2A traffic, keeping it separate from HTTP traffic (8080) |
| Agent card | JSON manifest at `/.well-known/agent-card.json` advertising the agent's capabilities |

**How discovery works:** The Orchestrator Agent (Notebook 3) uses `A2AClientToolProvider` to fetch this agent's card, discover available tools, and route product-related queries via A2A messages.

### Code Structure

The following cell creates `product_agent/a2a_server.py`:

| Section | What It Does |
|---------|--------------|
| Product Catalog Tools | Three `@tool` functions for search, category browse, and listing |
| Callback Handler | Log tool invocations to CloudWatch for debugging |
| Agent Initialization | Configure Strands agent with model, tools, and system prompt |
| A2A Server | Expose agent on port 9000 and generate agent card for discovery |

In [ ]:
%%writefile product_agent/a2a_server.py
"""Product Agent deployed to Amazon Bedrock AgentCore with A2A protocol support.

Queries product catalog from DummyJSON API (https://dummyjson.com) using custom
tools built with the Strands Agents @tool decorator.

Key concepts demonstrated:
- A2A protocol server for multi-agent orchestration (port 9000)
- Custom HTTP request tools for external API access
- OpenTelemetry instrumentation for AWS X-Ray distributed tracing
"""

import json
import logging
import os
from typing import Any

import boto3
import requests
import uvicorn
from fastapi import FastAPI
from strands import Agent, tool
from strands.models import BedrockModel
from strands.multiagent.a2a import A2AServer
from strands.telemetry import StrandsTelemetry

# --- Logging ---
logging.basicConfig(level=logging.INFO)
logging.getLogger("strands").setLevel(logging.INFO)
logger = logging.getLogger(__name__)

# Enable distributed tracing with AWS X-Ray
StrandsTelemetry().setup_otlp_exporter()

# --- Configuration ---
PORT = 9000  # A2A protocol uses port 9000 (HTTP protocol uses 8080)
SSM_PRODUCT_AGENT_URL = "/ecommerce-assistant/product-agent-url"

SYSTEM_PROMPT = """You are a Product Agent for an e-commerce assistant.

You have access to tools that query a product catalog with 194+ products across multiple categories.

Available tools:
- search_products: Search for products by keyword (e.g., "laptop", "phone")
- get_products_by_category: Get products from specific categories
- get_all_products: Browse all available products

Electronics categories available:
- laptops (MacBook Pro, Dell XPS, ThinkPad, etc.)
- smartphones (iPhone, Samsung, Google Pixel, etc.)
- tablets (iPad, Samsung Tab, etc.)
- mobile-accessories (phone cases, chargers, etc.)

Other categories: beauty, fragrances, furniture, groceries, mens-shirts, womens-dresses, and more.

Help customers find products by searching, browsing categories, or filtering by price.
If a product isn't found, suggest similar alternatives from available categories.
"""

# --- Product Catalog Tools ---
@tool
def search_products(query: str, limit: int = 10) -> str:
    """Search for products by keyword across all categories.

    Args:
        query: Search term (e.g., "laptop", "phone", "MacBook")
        limit: Maximum number of products to return (default: 10)

    Returns:
        JSON string with matching products including id, title, price, category, description
    """
    try:
        response = requests.get(
            "https://dummyjson.com/products/search",
            params={"q": query, "limit": limit},
            timeout=10,
        )
        response.raise_for_status()
        data = response.json()

        # Simplify response to essential fields
        products = [
            {
                "id": p["id"],
                "title": p["title"],
                "price": p["price"],
                "category": p["category"],
                "description": p["description"],
                "rating": p.get("rating", "N/A"),
                "stock": p.get("stock", "N/A"),
            }
            for p in data.get("products", [])
        ]

        return json.dumps(
            {"products": products, "total": data.get("total", 0)}, indent=2
        )
    except Exception as e:
        return json.dumps({"error": f"Failed to search products: {str(e)}"})


@tool
def get_products_by_category(category: str, limit: int = 10) -> str:
    """Get products from a specific category.

    Args:
        category: Category name (e.g., "laptops", "smartphones", "tablets", "beauty")
        limit: Maximum number of products to return (default: 10)

    Returns:
        JSON string with products from the specified category
    """
    try:
        response = requests.get(
            f"https://dummyjson.com/products/category/{category}",
            params={"limit": limit},
            timeout=10,
        )
        response.raise_for_status()
        data = response.json()

        # Simplify response to essential fields
        products = [
            {
                "id": p["id"],
                "title": p["title"],
                "price": p["price"],
                "category": p["category"],
                "description": p["description"],
                "rating": p.get("rating", "N/A"),
                "stock": p.get("stock", "N/A"),
            }
            for p in data.get("products", [])
        ]

        return json.dumps(
            {"products": products, "total": data.get("total", 0), "category": category},
            indent=2,
        )
    except Exception as e:
        return json.dumps(
            {"error": f"Failed to get products from category '{category}': {str(e)}"}
        )


@tool
def get_all_products(limit: int = 10) -> str:
    """Get all available products across all categories.

    Args:
        limit: Maximum number of products to return (default: 10)

    Returns:
        JSON string with products from all categories
    """
    try:
        response = requests.get(
            "https://dummyjson.com/products",
            params={"limit": limit},
            timeout=10,
        )
        response.raise_for_status()
        data = response.json()

        # Simplify response to essential fields
        products = [
            {
                "id": p["id"],
                "title": p["title"],
                "price": p["price"],
                "category": p["category"],
                "description": p["description"],
                "rating": p.get("rating", "N/A"),
                "stock": p.get("stock", "N/A"),
            }
            for p in data.get("products", [])
        ]

        return json.dumps(
            {"products": products, "total": data.get("total", 0)}, indent=2
        )
    except Exception as e:
        return json.dumps({"error": f"Failed to get all products: {str(e)}"})


# --- Callback Handler ---
class ToolLoggingHandler:
    """Log tool invocations to CloudWatch for debugging multi-agent workflows.

    Tracks which tools the agent calls and with what inputs. Useful for:
    - Debugging why an agent gave a certain response
    - Monitoring A2A calls in distributed systems
    - Building observability dashboards
    """

    def __init__(self) -> None:
        self.logged_tool_ids: set[str] = set()  # Prevent duplicate logs
        self.tool_count = 0

    def __call__(self, **kwargs: Any) -> None:
        message = kwargs.get("message", {})
        if isinstance(message, dict) and message.get("role") == "assistant":
            for content in message.get("content", []):
                if isinstance(content, dict):
                    tool_use = content.get("toolUse")
                    if tool_use:
                        tool_id = tool_use.get("toolUseId")
                        if tool_id and tool_id not in self.logged_tool_ids:
                            self.logged_tool_ids.add(tool_id)
                            self.tool_count += 1
                            tool_name = tool_use.get("name", "Unknown")
                            tool_input = tool_use.get("input", {})
                            logger.info(f"=== TOOL #{self.tool_count}: {tool_name} ===")
                            logger.info(f"TOOL INPUT: {json.dumps(tool_input)}")

        if kwargs.get("complete") and kwargs.get("data"):
            logger.info(f"=== COMPLETE: {len(kwargs.get('data', ''))} chars ===")


# --- Helper Functions ---
def get_runtime_url() -> str | None:
    """Get AgentCore runtime URL for agent card generation.

    The runtime URL is needed for the A2A agent card (/.well-known/agent-card.json)
    so other agents can discover and call this agent. Without it, the agent card
    will have an internal URL that won't work for external A2A calls.
    """
    if url := os.environ.get("PRODUCT_AGENT_URL"):
        logger.info(f"Using runtime URL from env: {url}")
        return url

    try:
        ssm = boto3.client("ssm")
        response = ssm.get_parameter(Name=SSM_PRODUCT_AGENT_URL, WithDecryption=True)
        url = response["Parameter"]["Value"]
        logger.info(f"Using runtime URL from SSM: {url}")
        return url
    except Exception as e:
        logger.warning(f"Could not get runtime URL: {e}")
        return None


# --- Agent and Server Setup ---
region = boto3.session.Session().region_name or "us-west-2"

product_agent = Agent(
    name="Ecommerce_Product_Agent",
    description="Fetches product information from DummyJSON catalog for browsing and discovery",
    system_prompt=SYSTEM_PROMPT,
    model=BedrockModel(
        model_id="us.anthropic.claude-sonnet-4-20250514-v1:0",
        region_name=region,
    ),
    tools=[search_products, get_products_by_category, get_all_products],
    callback_handler=ToolLoggingHandler(),
)
logger.info(f"Agent created: {product_agent.name}")

runtime_url = get_runtime_url()

a2a_server = A2AServer(
    agent=product_agent,
    host="0.0.0.0",  # Listen on all interfaces
    port=PORT,
    http_url=runtime_url,  # Used in agent card for A2A discovery
    serve_at_root=True,    # Serve agent card at /.well-known/agent-card.json
)

app = FastAPI(title="Product Agent A2A Server")


@app.get("/ping")
def ping():
    """Health check endpoint for AgentCore container probes."""
    return {"status": "healthy"}


app.mount("/", a2a_server.to_fastapi_app())

if __name__ == "__main__":
    uvicorn.run(app, host="0.0.0.0", port=PORT)

In [ ]:
%%writefile product_agent/requirements.txt
strands-agents[a2a,otel]==1.29.0
fastapi==0.115.0
uvicorn==0.34.0
boto3==1.42.60
requests==2.32.3

---
## Step 2: Deploy to Amazon Bedrock AgentCore

The `bedrock-agentcore-starter-toolkit` handles the deployment pipeline. Behind the scenes, it:

1. **Creates IAM role** — Grants permissions for ECR image pull, Bedrock model invocation, and CloudWatch logging
2. **Configures runtime** — Packages agent code with dependencies and sets A2A protocol
3. **Builds container** — Creates Docker image and pushes to Amazon ECR
4. **Launches runtime** — Starts managed container in Amazon Bedrock AgentCore
5. **Stores URL** — Saves runtime endpoint to Parameter Store for discovery

### What Gets Created

| Resource | Name/Path | Purpose |
|----------|-----------|---------|
| IAM Role | `ecommerce-product-agent-role` | Grants the agent runtime permissions to pull container images from Amazon ECR, invoke Amazon Bedrock models, and write logs to Amazon CloudWatch |
| ECR Repository | `ecommerce_product_agent` | Stores the Docker container image that Amazon Bedrock AgentCore pulls to run the agent |
| AgentCore Runtime | `ecommerce_product_agent` | Managed compute environment that runs your containerized A2A server and handles scaling, networking, and health monitoring |
| SSM Parameter | `/ecommerce-assistant/product-agent-url` | Stores the runtime invocation URL so the Orchestrator and agent container can discover this agent's endpoint |

### Create IAM Role and Configure Runtime

The IAM execution role allows the AgentCore runtime to pull container images, invoke Bedrock models, and write logs. The `Runtime.configure()` method packages your agent code for deployment.

| Parameter | Value | Purpose |
|-----------|-------|---------|
| `entrypoint` | `a2a_server.py` | Python file that starts the A2A server |
| `protocol` | `A2A` | Enables port 9000 for agent-to-agent messaging |
| `auto_create_ecr` | `True` | Creates ECR repository if it doesn't exist |

In [ ]:
from bedrock_agentcore_starter_toolkit import Runtime

# Create IAM execution role for Product Agent
product_role_arn = create_agentcore_role(PRODUCT_ROLE_NAME, account_id, region)
print(f"IAM Role ARN: {product_role_arn}")

# Change to product_agent directory
product_agent_dir = NOTEBOOK_DIR / "product_agent"
os.chdir(product_agent_dir)

# Configure runtime
product_runtime = Runtime()
product_runtime.configure(
    entrypoint="a2a_server.py",
    execution_role=product_role_arn,
    auto_create_ecr=True,
    requirements_file="requirements.txt",
    region=region,
    agent_name=PRODUCT_AGENT_NAME,
    protocol="A2A",
)
print("Product Agent runtime configured")

### Launch Agent

`Runtime.launch()` builds the Docker image, pushes it to ECR, and creates the AgentCore runtime. The `auto_update_on_conflict=True` flag updates an existing runtime if one already exists with the same name.

**Note:** First deployment takes 5-10 minutes (subsequent updates are faster).

In [ ]:
# Launch Product Agent
print("Launching Product Agent (this may take several minutes)...")
product_launch = product_runtime.launch(auto_update_on_conflict=True)
print(f"Product Agent ARN: {product_launch.agent_arn}")
PRODUCT_AGENT_ARN = product_launch.agent_arn

### Get Runtime URL

Once the runtime reaches `ACTIVE` or `READY` status, construct the invocation URL from the runtime ARN. This URL is the HTTPS endpoint that the Orchestrator will use to send A2A messages to this agent.

In [ ]:
# Get runtime status and URL
status_response = product_runtime.status()
status = status_response.endpoint.get("status", "")
product_agent_url = None  # Initialize before conditional

print(f"Product Agent Status: {status}")

if status.upper() in ["ACTIVE", "READY"]:
    agent_runtime_arn = status_response.endpoint.get("agentRuntimeArn")
    escaped_arn = quote(agent_runtime_arn, safe="")
    product_agent_url = f"https://bedrock-agentcore.{region}.amazonaws.com/runtimes/{escaped_arn}/invocations"
    print(f"Product Agent URL: {product_agent_url}")
else:
    print(f"WARNING: Product Agent status is {status}")
    print("Expected status: ACTIVE or READY")

### Store Runtime URL in Parameter Store

Store the URL in AWS Systems Manager Parameter Store so other components can discover this agent:
- **Agent container** reads it at startup to populate the agent card's `http_url` field
- **Orchestrator Agent** (Notebook 3) retrieves it to configure `A2AClientToolProvider`

In [ ]:
# Store Product Agent URL in SSM
if status.upper() in ["ACTIVE", "READY"]:
    result = store_ssm_parameter(
        param_name=SSM_PRODUCT_AGENT_URL,
        value=product_agent_url,
        region=region
    )
    print(result["message"])
    print(f"Parameter version: {result['version']}")
else:
    print("Skipping SSM storage - agent not ready")

# Return to notebook directory
os.chdir(NOTEBOOK_DIR)

In [ ]:
print("=" * 60)
print("Product Agent Deployment Summary")
print("=" * 60)
print(f"Agent Name: {PRODUCT_AGENT_NAME}")
print(f"Agent ARN: {PRODUCT_AGENT_ARN}")
print(f"IAM Role: {product_role_arn}")
print(f"Runtime URL: {product_agent_url or 'Not available - agent not ready'}")
print(f"SSM Parameter: {SSM_PRODUCT_AGENT_URL}")
print(f"Status: {status}")
print(f"Region: {region}")
print(f"Protocol: A2A (port 9000)")
print("=" * 60)

---
## Step 3: Test Product Agent

Verify the agent works standalone before integrating with the Orchestrator. The `Runtime.invoke()` method calls the AgentCore runtime directly—useful for testing individual agents without needing another agent to send A2A messages.

**Example queries to try:**
- `"Show me laptops under $1000"`
- `"Search for wireless headphones"`
- `"What smartphones are available?"`

**Expected response:**
- Agent invokes `search_products` or `get_products_by_category` tool
- Returns JSON with matching products (id, title, price, category)
- Response is conversational, summarizing the product results

In [ ]:
# Test Product Agent with A2A protocol payload
test_message = "Show me electronics under $100"

# A2A protocol uses JSON-RPC 2.0 format
payload = {
    "jsonrpc": "2.0",
    "method": "message/send",
    "id": str(uuid4()),
    "params": {
        "message": {
            "messageId": str(uuid4()),
            "role": "user",
            "parts": [{"kind": "text", "text": test_message}]
        }
    }
}

print("Testing Product Agent...")
print(f"Query: {test_message}")
print("-" * 60)

try:
    response = product_runtime.invoke(payload=payload, session_id=str(uuid4()))
    if "result" in str(response) and "artifacts" in str(response):
        print(response)
    else:
        print("Unexpected response format")
        print(str(response)[:500])
except Exception as e:
    print(f"Error: {e}")

### Verify Agent Card (A2A Discovery)

The `A2AServer` wrapper automatically generates an **A2A-compliant agent card** from your Strands agent. This card is served at `/.well-known/agent-card.json` and enables other agents to discover this agent's capabilities at runtime.

**What A2AServer auto-generates:**
- Agent name and description from `Agent()` constructor
- Skills array from `@tool` decorated functions
- Supported input/output modes and transport protocol
- Runtime URL for A2A messaging

In [ ]:
# Fetch the auto-generated agent card
from botocore.auth import SigV4Auth
from botocore.awsrequest import AWSRequest
import requests
import json

agent_card_url = f"{product_agent_url}/.well-known/agent-card.json"
credentials = boto3.Session().get_credentials()
request = AWSRequest(method="GET", url=agent_card_url)
SigV4Auth(credentials, "bedrock-agentcore", region).add_auth(request)

response = requests.get(agent_card_url, headers=dict(request.headers))
agent_card = response.json()

print(json.dumps(agent_card, indent=2))

---
## Next Steps

| Notebook | What You'll Build |
|----------|-------------------|
| **2. Deploy Order Agent** | DynamoDB integration with MCP tools, async lifespan pattern for slow-starting dependencies |
| **3. Deploy Orchestrator** | HTTP-facing agent that routes requests to Product and Order agents via `A2AClientToolProvider` |

---
## Cleanup (Optional)

Run this section to delete all resources created by this notebook.

In [ ]:
# Delete SSM parameter
ssm = boto3.client("ssm", region_name=region)
try:
    ssm.delete_parameter(Name=SSM_PRODUCT_AGENT_URL)
    print(f"Deleted SSM parameter: {SSM_PRODUCT_AGENT_URL}")
except Exception as e:
    print(f"Error deleting SSM parameter: {e}")

# Delete auto-generated toolkit files
print("\nDeleting auto-generated files...")
for filename in [".bedrock_agentcore.yaml", "Dockerfile", ".dockerignore"]:
    filepath = product_agent_dir / filename
    if filepath.exists():
        filepath.unlink()
        print(f"  Deleted: {filename}")

print("\nCleanup complete!")